In [1]:
%load_ext autoreload
%autoreload 2

import src

Cargamos los siguientes archivos:

1. `ENOE_VIVT123.csv`, que contiene las	viviendas de la ENOE y tiene las características y llaves de vivienda.
2. `ENOE_HOGT123.csv`, que contiene los	hogares de la ENOE y sirve para la construcción de hogares.
3. `ENOE_SDEMT123.csv`, que contiene los datos sociodemográficos de la ENOE (Edad, sexo, relación familiar, empleo, ponderadores, etc).
4. `ENOE_COE1T123.csv`, que tiene el cuestionario de ocupación I con las características laborales.
5. `ENOE_COE2T123.csv`, que tiene el cuestionario de ocupación II con la información laboral complementaria.
6. `IMEPLAN_Base_Habitantes_Master.csv`, con las personas de la encuesta OD, mostrando los individuos y atributos sociodemográficos.
7. `IMEPLAN_Base_Viviendas_Master.csv`, con las viviendas de la encuesta OD, mostrando las características del hogar/vivienda.
8. `IMEPLAN_Base_Viajes_Master.csv`, con los viajes de la encuesta OD, con datos como el origen, destino, propósito, modo, etc.

In [2]:
enoe_housing, enoe_households, enoe_sociodemographic, enoe_questionnaire_1, enoe_questionnaire_2 = src.load_enoe_tables("data/enoe")

od_population = src.load_od_population("data/od")
od_households = src.load_od_households("data/od")
od_trips = src.load_od_trips("data/od")

Para la ENOE juntamos las 5 tablas previamente importadas usando los atributos de vivienda, hogar y persona. Además calculamos el tamaño de vivienda, filtramos a personas ocupadas de Jalisco y conservamos las variables laborales y los ponderadores necesarios para el posterior análisis.

Para la OD, juntamos la base de habitantes con la de viviendas de la OD, incorporamos el tamaño del hogar y filtramos a las personas ocupadas.

In [3]:
enoe = src.generate_enoe_dataframe(enoe_housing, enoe_households, enoe_sociodemographic, enoe_questionnaire_1, enoe_questionnaire_2)
od = src.generate_od_dataframe(od_population, od_households)

In [6]:
print(f"ENOE workers: {enoe.shape}")
print(f"OD workers: {od.shape}")

assert enoe["ent"].eq(14).all(), "ENOE contains observations outside Jalisco."
assert enoe["survey_weight"].notna().all(), "ENOE contains missing survey weights."
assert od["expansion_factor"].notna().all(), "OD contains missing expansion factors."

display(enoe.head())
display(od.head())

ENOE workers: (6793, 22)
OD workers: (26913, 60)


,tipo,mes_cal,cd_a,ent,con,v_sel,n_hog,h_mud,n_ren,mun,...,survey_weight,sex,pos_ocu,scian,eda,cs_p13_1,emp_ppal,e_con,par_c,dwelling_size
0,2,1,2,14,2929,1,1,0,1,97,...,665,1,1,18,29,4,1,6,101,4
1,2,1,2,14,2929,1,1,0,2,97,...,665,1,1,5,21,4,2,6,403,4
2,2,1,2,14,2929,1,1,0,3,97,...,665,2,1,5,20,4,2,6,403,4
3,2,1,2,14,2929,1,1,0,4,97,...,665,2,1,5,17,4,1,6,403,4
4,2,1,2,14,2929,2,1,0,1,97,...,665,1,2,5,44,3,2,5,101,4


,Folio Vivienda,Folio Habitante,Fecha,Municipio,AGEB,Centralidad,¿Salió de su casa ayer?,¿Cuál es la razón por la que NO realizó algún viaje?,Viajes contados,Día de la semana que realizó los viajes:,...,¿Cuáles son los principales destinos de los viajes en fin de semana? | Deportes o recreación,¿Cuáles son los principales destinos de los viajes en fin de semana? | Llevar o recoger a alguien,¿Cuáles son los principales destinos de los viajes en fin de semana? | Hacer un trámite,¿Cuáles son los principales destinos de los viajes en fin de semana? | Al médico o atención de salud,¿Cuáles son los principales destinos de los viajes en fin de semana? | Pagar algún servicio o al banco,¿Cuáles son los principales destinos de los viajes en fin de semana? | Guardería,¿Cuáles son los principales destinos de los viajes en fin de semana? | Regresar a Casa,¿Cuáles son los principales destinos de los viajes en fin de semana? | Religión,expansion_factor,dwelling_size
0,1,1,25-Jan-23,Tlajomulco,1409700251418,30F,Sí,NaN,7,Jueves,...,Sí,Sí,Sí,Sí,Sí,Sí,Sí,No,51,5
1,2,1,26-Jan-23,Guadalajara,1403900010859,09,Sí,NaN,2,Miércoles,...,Sí,No,No,No,No,No,No,No,249,3
2,2,2,26-Jan-23,Guadalajara,1403900010859,09,No,Otros (especifique),0,NaN,...,No,No,No,No,No,No,No,No,249,3
3,2,3,26-Jan-23,Guadalajara,1403900010859,09,No,Otros (especifique),0,NaN,...,No,No,No,No,No,No,No,No,249,3
4,3,1,26-Jan-23,Guadalajara,1403900010539,09,No,Otros (especifique),0,NaN,...,No,No,No,No,No,No,No,No,249,3


In [7]:
from pathlib import Path

In [8]:
output_directory = Path("outputs")
output_directory.mkdir(exist_ok=True)

enoe.to_parquet(output_directory / "enoe_workers.parquet", index=False)
od.to_parquet(output_directory / "od_workers.parquet", index=False)